# KPoEM Emotion Distribution Visualization

* 본 코드는 KPoEM 데이터셋에 포함된 개별 시인의 작품을 대상으로 감정 분포를 시각화하기 위한 코드이다.
* 입력 데이터는 KPoEM 데이터셋의 감정 라벨링 결과를 기반으로 하며, 각 감정의 출현 빈도를 집계하여 시각화한다.
* 시각화 결과는 특정 시인의 작품 세계에서 두드러지게 나타나는 감정적 경향과 분포를 탐색하는 데 활용될 수 있다.
* 본 코드는 김소월, 윤동주, 이상, 임화, 한용운 등 KPoEM에 포함된 시인별 감정 분포 분석에 공통적으로 적용 가능하도록 설계되었다.
* 생성된 시각화는 KPoEM 플랫폼의 Distant Reading 기능에서 활용된다.

In [1]:
import pandas as pd
import plotly.express as px
from collections import Counter

In [2]:
url = "https://huggingface.co/datasets/AKS-DHLAB/KPoEM/resolve/main/KPoEM_line_dataset_v4.tsv"

df = pd.read_csv(
    url,
    sep="\t",
    encoding="utf-8",
    quoting=3
)

df.head()


,line_id,poem_id,text,sub_title,title,poet,annotator_01,annotator_02,annotator_03,annotator_04,annotator_05
0,1,1,죽는 날까지 하늘을 우러러,NaN,서시,윤동주,비장함,비장함,"뿌듯함, 비장함","비장함, 뿌듯함, 감동/감탄","비장함, 서러움, 슬픔"
1,2,1,"한 점 부끄럼이 없기를,",NaN,서시,윤동주,"부끄러움, 비장함","부끄러움, 비장함, 기대감, 불안/걱정, 서러움, 슬픔","깨달음, 비장함, 뿌듯함","비장함, 부끄러움, 기대감",비장함
2,3,1,잎새에 이는 바람에도,NaN,서시,윤동주,"기대감, 신기함/관심","기대감, 불안/걱정, 비장함","슬픔, 서러움, 불안/걱정, 당황/난처","비장함, 슬픔","감동/감탄, 신기함/관심, 편안/쾌적, 기대감"
3,4,1,나는 괴로워했다.,NaN,서시,윤동주,"절망, 슬픔, 패배/자기혐오","절망, 슬픔, 패배/자기혐오, 죄책감, 힘듦/지침, 비장함","당황/난처, 서러움, 죄책감, 패배/자기혐오","비장함, 슬픔, 패배/자기혐오, 절망, 힘듦/지침","슬픔, 서러움, 절망, 힘듦/지침, 패배/자기혐오"
4,5,1,별을 노래하는 마음으로,NaN,서시,윤동주,"기쁨, 신기함/관심, 즐거움/신남, 흐뭇함(귀여움/예쁨), 뿌듯함","기쁨, 뿌듯함, 감동/감탄, 슬픔, 비장함, 아껴주는, 환영/호의, 기대감","고마움, 기대감, 기쁨, 아껴주는, 흐뭇함(귀여움/예쁨)","감동/감탄, 기대감, 기쁨, 아껴주는, 행복","즐거움/신남, 기대감, 기쁨, 행복"


In [3]:
url = "https://huggingface.co/datasets/AKS-DHLAB/KPoEM/resolve/main/KPoEM_poem_dataset_v4.tsv"

df_poem = pd.read_csv(
    url,
    sep="\t",
    encoding="utf-8",
    quoting=3
)

df_poem.head(10)


,seg_id,poem_id,text,sub_title,title,poetry_book,poet,annotator_01,annotator_02,annotator_03,annotator_04,annotator_05
0,1,1,"죽는 날까지 하늘을 우러러 한 점 부끄럼이 없기를, 잎새에 이는 바람에도 나는 괴로...",NaN,서시,하늘과 바람과 별과 시,윤동주,"불안/걱정, 비장함, 서러움, 슬픔, 안타까움/실망, 부끄러움, 죄책감, 패배/자기혐오","패배/자기혐오, 서러움, 비장함, 부끄러움, 아껴주는, 슬픔, 불안/걱정, 죄책감","깨달음, 흐뭇함(귀여움/예쁨), 고마움, 힘듦/지침, 안타까움/실망","비장함, 뿌듯함, 슬픔","비장함, 감동/감탄, 깨달음, 서러움"
1,2,2,산모퉁이를 돌아 논가 외딴 우물을 홀로 찾아가선 가만히 들여다봅니다. 우물 속에는 ...,NaN,자화상,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 아껴주는, 신기함/관심, 흐뭇함(귀여움/예쁨), 서러움, 짜증, 지긋...","신기함/관심, 증오/혐오, 불쌍함/연민, 화남/분노, 서러움, 슬픔, 아껴주는, 패...","안타까움/실망, 서러움, 슬픔, 깨달음, 흐뭇함(귀여움/예쁨)","불평/불만, 안타까움/실망, 의심/불신, 슬픔","불쌍함/연민, 불안/걱정"
2,3,3,쫓아오든 햇빛인데 지금 교회당 꼭대기 십자가에 걸리었습니다. 첨탑(尖塔)이 저렇게...,NaN,십자가,하늘과 바람과 별과 시,윤동주,"서러움, 패배/자기혐오, 존경, 감동/감탄, 비장함, 불쌍함/연민, 불안/걱정, 죄책감","감동/감탄, 놀람, 불쌍함/연민, 비장함, 슬픔, 행복, 아껴주는, 불안/걱정","절망, 깨달음, 서러움, 힘듦/지침, 존경, 비장함","당황/난처, 힘듦/지침, 놀람, 슬픔, 불안/걱정","깨달음, 비장함, 존경, 신기함/관심"
3,4,4,바람이 어디로부터 불어 와 어디로 불려 가는 것일까 바람이 부는데 내 괴로움에는 ...,NaN,바람이 불어,하늘과 바람과 별과 시,윤동주,"슬픔, 서러움, 한심함, 패배/자기혐오, 죄책감, 부끄러움","비장함, 서러움, 슬픔, 불안/걱정, 패배/자기혐오, 죄책감","힘듦/지침, 안타까움/실망, 서러움, 절망, 깨달음","힘듦/지침, 당황/난처, 불안/걱정, 불평/불만, 슬픔, 부끄러움, 패배/자기혐오","깨달음, 불쌍함/연민, 불안/걱정, 서러움, 패배/자기혐오, 힘듦/지침"
4,5,5,고향에 돌아온 날 밤에 내 백골(白骨)이 따라와 한방에 누웠다. 어둔 방은 우주로...,NaN,또 다른 고향,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 비장함, 공포/무서움, 슬픔, 서러움, 의심/불신, 깨달음, 기대감","힘듦/지침, 놀람, 서러움, 슬픔, 감동/감탄, 존경, 죄책감, 패배/자기혐오, 비...","절망, 깨달음, 서러움, 힘듦/지침, 안타까움/실망","슬픔, 힘듦/지침, 절망, 기대감, 안타까움/실망, 불안/걱정","공포/무서움, 당황/난처, 놀람, 불안/걱정, 비장함"
5,6,6,계절이 지나가는 하늘에는 가을로 가득 차 있습니다. 나는 아무 걱정도 없이 가을 ...,NaN,별 헤는 밤,하늘과 바람과 별과 시,윤동주,"흐뭇함(귀여움/예쁨), 감동/감탄, 아껴주는, 슬픔, 기쁨, 기대감, 깨달음, 환영...","감동/감탄, 기대감, 서러움, 슬픔, 비장함, 아껴주는, 흐뭇함(귀여움/예쁨), 안...","흐뭇함(귀여움/예쁨), 기대감, 안타까움/실망, 서러움, 슬픔","감동/감탄, 기쁨, 흐뭇함(귀여움/예쁨), 행복, 아껴주는, 편안/쾌적","존경, 감동/감탄, 신기함/관심, 깨달음, 서러움, 슬픔, 안타까움/실망"
6,7,6,"어머님, 나는 별 하나에 아름다운 말 한 마디씩 불러 봅니다. 소학교 때 책상을 같...",NaN,별 헤는 밤,하늘과 바람과 별과 시,윤동주,"아껴주는, 슬픔, 부끄러움, 불쌍함/연민, 존경, 감동/감탄, 고마움, 흐뭇함(귀여...","아껴주는, 흐뭇함(귀여움/예쁨), 환영/호의, 서러움, 슬픔, 깨달음, 죄책감, 부...","안타까움/실망, 서러움, 기대감, 슬픔, 깨달음","불안/걱정, 서러움, 슬픔, 존경, 환영/호의","서러움, 슬픔, 존경"
7,8,7,"살구나무 그늘로 얼굴을 가리고, 병원 뒤뜰에 누워, 젊은 여자가 흰 옷 아래로 하얀...",NaN,병원,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 슬픔, 서러움, 안타까움/실망, 힘듦/지침, 패배/자기혐오, 부담/안...","신기함/관심, 불쌍함/연민, 서러움, 슬픔, 안타까움/실망, 당황/난처, 힘듦/지침...","안타까움/실망, 서러움, 힘듦/지침, 고마움, 기대감","서러움, 불안/걱정, 불평/불만, 화남/분노, 의심/불신, 증오/혐오","신기함/관심, 불안/걱정, 불평/불만, 불쌍함/연민"
8,9,8,잃어버렸습니다. 무얼 어디다 잃었는지 몰라 두 손의 호주머니를 더듬어 길에 나...,NaN,길,하늘과 바람과 별과 시,윤동주,"당황/난처, 슬픔, 안타까움/실망, 부끄러움, 부담/안_내킴, 깨달음, 패배/자기혐...","당황/난처, 불안/걱정, 놀람, 서러움, 슬픔, 부끄러움, 감동/감탄, 죄책감, 깨...","안타까움/실망, 서러움, 슬픔, 깨달음, 힘듦/지침","당황/난처, 불안/걱정, 슬픔, 힘듦/지침","불안/걱정, 비장함, 당황/난처, 깨달음"
9,10,9,세상으로부터 돌아오듯이 이제 내 좁은방에 돌아와 불을 끄옵니다. 불을 켜 두는 것은...,NaN,돌아와 보는 밤,하늘과 바람과 별과 시,윤동주,"힘듦/지침, 서러움, 기대감, 깨달음, 슬픔, 비장함","힘듦/지침, 불안/걱정, 부담/안_내킴, 공포/무서움, 불쌍함/연민, 서러움, 슬픔...","힘듦/지침, 깨달음, 편안/쾌적, 슬픔, 불안/걱정","서러움, 슬픔, 힘듦/지침, 불안/걱정","힘듦/지침, 서러움, 깨달음"


In [36]:
poet = "한용운"

In [37]:
# 시인별 작품 데이터
df_poem_kim = df_poem[df_poem['poet'] == poet]

In [38]:
# 시인별 라인 데이터
df_kim = df[df["poet"] == poet].copy()

In [39]:
## 시인별 전체 데이터(행+작품)
df_merged = pd.concat([df_kim, df_poem_kim], axis=0, ignore_index=True)

In [40]:
df_merged

,line_id,poem_id,text,sub_title,title,poet,annotator_01,annotator_02,annotator_03,annotator_04,annotator_05,seg_id,poetry_book
0,2822.0,194,"임은 갔습니다. 아아, 사랑하는 나의 임은 갔습니다.",NaN,님의 침묵,한용운,"슬픔, 서러움, 안타까움/실망","슬픔, 서러움, 안타까움/실망, 불안/걱정, 절망","경악, 당황/난처, 부담/안_내킴, 불쌍함/연민, 비장함, 서러움, 슬픔, 안타까움/실망","당황/난처, 안타까움/실망, 슬픔, 서러움, 경악, 아껴주는","불안/걱정, 슬픔, 서러움, 절망, 힘듦/지침",NaN,NaN
1,2823.0,194,푸른 산빛을 깨치고 단풍나무 숲을 향하여 난 작은 길을 걸어서 차마 떨치고 갔습니다.,NaN,님의 침묵,한용운,"비장함, 슬픔, 당황/난처","비장함, 슬픔, 서러움, 불안/걱정, 부담/안_내킴, 안타까움/실망, 패배/자기혐오","부담/안_내킴, 불평/불만, 비장함, 서러움, 안타까움/실망","비장함, 뿌듯함, 불안/걱정","비장함, 힘듦/지침",NaN,NaN
2,2824.0,194,황금의 꽃같이 굳고 빛나던 옛 맹세는 차디찬 티끌이 되어서 한숨의 미풍에 날아갔습니다.,NaN,님의 침묵,한용운,"비장함, 슬픔, 어이없음, 서러움, 안타까움/실망","비장함, 서러움, 슬픔, 안타까움/실망, 절망, 부끄러움","당황/난처, 불쌍함/연민, 부담/안_내킴, 안타까움/실망","슬픔, 안타까움/실망, 의심/불신, 절망, 힘듦/지침","서러움, 안타까움/실망, 패배/자기혐오",NaN,NaN
3,2825.0,194,날카로운 첫키스의 추억은 나의 운명의 지침을 돌려 놓고 뒷걸음쳐서 사라졌습니다.,NaN,님의 침묵,한용운,"안타까움/실망, 불안/걱정, 깨달음","깨달음, 놀람, 비장함, 서러움, 기쁨, 슬픔","기쁨, 깨달음, 비장함, 안타까움/실망","놀람, 당황/난처, 슬픔, 안타까움/실망","기쁨, 놀람, 슬픔, 서러움",NaN,NaN
4,2826.0,194,나는 향기로운 임의 말소리에 귀먹고 꽃다운 임의 얼굴에 눈멀었습니다.,NaN,님의 침묵,한용운,"아껴주는, 흐뭇함(귀여움/예쁨), 환영/호의","아껴주는, 흐뭇함(귀여움/예쁨), 감동/감탄, 행복","기대감, 기쁨, 감동/감탄, 아껴주는, 존경, 행복","깨달음, 비장함, 슬픔, 행복, 흐뭇함(귀여움/예쁨), 아껴주는","당황/난처, 슬픔, 증오/혐오",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1331,NaN,445,물보다 깊으니라 갈산(추산(秋山))보다 높으니라 달보다 빛나리라 돌보다 굳으리라 사...,NaN,사랑,한용운,"감동/감탄, 고마움, 비장함, 깨달음, 아껴주는, 기대감, 환영/호의","감동/감탄, 깨달음, 기쁨, 기대감, 비장함, 뿌듯함, 아껴주는, 행복, 환영/호의...","감동/감탄, 기대감, 깨달음, 행복","감동/감탄, 기쁨, 뿌듯함, 환영/호의, 행복, 즐거움/신남","비장함, 뿌듯함, 깨달음",565.0,NaN
1332,NaN,446,백리(百里)를 갈 양이면 구십리(九十里)가 반(半)이라네 시작(始作)이 반(半)이라...,NaN,성공,한용운,"안타까움/실망, 부담/안_내킴, 불평/불만, 불안/걱정, 의심/불신, 힘듦/지침, 짜증","깨달음, 힘듦/지침, 부담/안_내킴, 불안/걱정, 불평/불만, 안타까움/실망, 지긋지긋","깨달음, 불평/불만, 안타까움/실망, 한심함","감동/감탄, 기대감, 뿌듯함, 즐거움/신남","깨달음, 기쁨, 절망, 안타까움/실망",566.0,NaN
1333,NaN,447,사나이 되얏으니 무슨 일을 (하)야 볼까 밭을 팔아 책을 살까 책을 덮고 칼을 갈까...,NaN,남아,한용운,"비장함, 기대감, 깨달음","깨달음, 부담/안_내킴, 비장함, 서러움, 안타까움/실망","감동/감탄, 기대감, 깨달음, 뿌듯함, 비장함","감동/감탄, 기대감, 비장함, 뿌듯함","뿌듯함, 비장함, 신기함/관심, 기쁨",567.0,NaN
1334,NaN,448,이 작고 더럽고 밉살스런 파리야 너는 썩은 쥐인지 만두(饅頭)인지 분간을 못하는 더...,NaN,파리,한용운,"불쌍함/연민, 깨달음, 부끄러움, 우쭐댐/무시함, 죄책감, 패배/자기혐오, 서러움,...","패배/자기혐오, 한심함, 화남/분노, 증오/혐오, 깨달음, 불쌍함/연민, 역겨움/징...","깨달음, 불평/불만, 어이없음, 지긋지긋, 화남/분노","경악, 불평/불만, 증오/혐오, 화남/분노, 절망","역겨움/징그러움, 화남/분노, 증오/혐오, 깨달음, 경악",568.0,NaN


In [41]:
# Extract all annotator columns
annotator_columns = [f'annotator_0{i}' for i in range(1, 6)]
all_emotions = []

# Iterate through each row and each annotator column
for _, row in df_merged[annotator_columns].iterrows():
    for col in annotator_columns:
        if pd.notna(row[col]) and row[col] != '없음': # Check for NaN and '없음'
            # Split by comma and strip whitespace
            emotions = [e.strip() for e in row[col].split(',')]
            all_emotions.extend(emotions)

# Count the occurrences of each emotion
emotion_counts = Counter(all_emotions)

# Convert to DataFrame for Plotly
emotion_df = pd.DataFrame(emotion_counts.items(), columns=['Emotion', 'Count'])

# Sort by count for better visualization
emotion_df = emotion_df.sort_values(by='Count', ascending=False)

In [42]:
emotion_df

,Emotion,Count
0,슬픔,2080
10,아껴주는,2035
9,비장함,1799
2,안타까움/실망,1580
1,서러움,1549
3,불안/걱정,1460
18,깨달음,1443
25,기대감,1400
23,감동/감탄,986
20,기쁨,908


In [43]:
# Create a bar chart using Plotly Express
fig = px.bar(
    emotion_df,
    x='Emotion',
    y='Count',
    title=poet + ' 작품 감정 분포',
    labels={'Emotion': '감정', 'Count': '횟수'},
    color='Count',
    color_continuous_scale=px.colors.sequential.Plasma
)

# Update layout for better readability
fig.update_layout(
    xaxis_title='감정',
    yaxis_title='횟수',
    font=dict(family='Arial', size=12),
    bargap=0.2 # Space between bars
)

fig.update_xaxes(tickangle=45)

# Convert the plot to an HTML string
html_output = fig.to_html(include_plotlyjs='cdn')

# Print the HTML output
print(html_output)

<html>
<head><meta charset="utf-8" /></head>
<body>
    <div>                        <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-3.5.0.min.js" integrity="sha256-fHbNLP+GlIXN+efbQec78UkemUz3NJp7UmfGxC1tNxs=" crossorigin="anonymous"></script>                <div id="34bd32d2-1bb7-4b7c-b46a-57f48a1e89b5" class="plotly-graph-div" style="height:100%; width:100%;"></div>            <script>                window.PLOTLYENV=window.PLOTLYENV || {};                                if (document.getElementById("34bd32d2-1bb7-4b7c-b46a-57f48a1e89b5")) {                    Plotly.newPlot(                        "34bd32d2-1bb7-4b7c-b46a-57f48a1e89b5",                        [{"hovertemplate":"\uac10\uc815=%{x}\u003cbr\u003e\ud69f\uc218=%{marker.color}\u003cextra\u003e\u003c\u002fextra\u003e","legendgroup":"","marker":{"color":{"dtype":"i2","bdata":"IAjzBwcHLAYNBrQFowV4BdoDjAN7A2QDXgPTAskChgJrAmYCWQIRAvcB3gG5AaABSgEmAf